In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
%skip
%sql
-- SHOW CATALOGS; 

In [0]:
# %sql
# CREATE CATALOG IF NOT EXISTS catalog_project1; 
# USE CATALOG catalog_project1; 

In [0]:
%skip
%sql
-- SHOW SCHEMAS; 

In [0]:
# %sql
# CREATE SCHEMA IF NOT EXISTS schema_project1; 
# USE SCHEMA schema_project1; 

US: Ingest raw CSV files into the Bronze layer with metadata so that the original data is preserved in Delta format for downstream processing.

#### Verify the exact location by checking if the file exists:

In [0]:
%skip
import os

# Check different possible paths
paths_to_check = [
    "/Workspace/Users/krishnasumanbhaipatel@gmail.com/Practice Folder/lakehouse-practice-project/src",
    "/Users/krishnasumanbhaipatel@gmail.com/Practice Folder/lakehouse-practice-project/src",
    "/Workspace/Users/krishnasumanbhaipatel@gmail.com/Practice Folder/lakehouse-practice-project/src/utils/logger.py",
    "/Users/krishnasumanbhaipatel@gmail.com/Practice Folder/lakehouse-practice-project/src/utils/logger.py"
]

for path in paths_to_check:
    exists = os.path.exists(path)
    print(f"{exists}\t{path}")

The file exists at /Workspace/Users/krishnasumanbhaipatel@gmail.com/Practice Folder/lakehouse-practice-project/src/utils/logger.py. However, for Python to recognize src and utils as packages, they need __init__.py files. Check:

In [0]:
%skip
import os

base = "/Workspace/Users/krishnasumanbhaipatel@gmail.com/Practice Folder/lakehouse-practice-project"

# Check for __init__.py files
init_files = [
    os.path.join(base, "src", "__init__.py"),
    os.path.join(base, "src/utils", "__init__.py")
]

print("Required __init__.py files:")
for path in init_files:
    exists = os.path.exists(path)
    status = "✓ EXISTS" if exists else "✗ MISSING"
    print(f"{status}\t{path}")

Root cause found! The __init__.py files are missing. Python requires these files to recognize directories as packages. Create them:

In [0]:
%skip
import os

base = "/Workspace/Users/krishnasumanbhaipatel@gmail.com/Practice Folder/lakehouse-practice-project"

# Create __init__.py files
init_files = [
    os.path.join(base, "src", "__init__.py"),
    os.path.join(base, "src/utils", "__init__.py")
]

for path in init_files:
    with open(path, 'w') as f:
        f.write("# Package initializer\n")
    print(f"✓ Created: {path}")

Perfect! The __init__.py files have been created. Verify the import works directly: 

In [0]:
%skip
import sys
sys.path.append("/Workspace/Users/krishnasumanbhaipatel@gmail.com/Practice Folder/lakehouse-practice-project")

# Try the import
try:
    from src.utils.logger import get_logger
    print("✓ Import successful!")
    logger = get_logger("test")
    print(f"✓ Logger created: {logger}")
except Exception as e:
    print(f"✗ Import failed: {e}")
    import traceback
    traceback.print_exc()

Great! The import works perfectly in a fresh execution. The issue is that Cell 7's cached execution state is stale. Let me restart the Python kernel to clear the cache, then run Cell 7: 

In [0]:
# dbutils.library.restartPython()

#### Check existing volumns in bronze schema 

In [0]:
%skip
%sql
-- SHOW VOLUMES IN catalog_project1.bronze;

Create a volumn for logs 

In [0]:
# %sql
# -- CREATE VOLUME IF NOT EXISTS catalog_project1.bronze.logs;
# spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}.{LOGS_VOLUME}")

Verify log file exists in UC (Unity Catalog) Volume 

In [0]:
%skip
dbutils.fs.ls("/Volumes/catalog_project1/bronze/logs/")

List files in log volume with display 

In [0]:
%skip
display(dbutils.fs.ls("/Volumes/catalog_project1/bronze/logs/"))

### Configuration

In [0]:
# import sys
# sys.path.append("/Workspace/Users/krishnasumanbhaipatel@gmail.com/Practice Folder/lakehouse-practice-project")

# import importlib

# from src.utils.audit import *
# from src.ingestion.bronze import add_metadata
# from src.utils.config import *

# importlib.reload(src.utils.audit)
# importlib.reload(src.ingestion.bronze)
# importlib.reload(src.utils.config)

  File /Workspace/Users/krishnasumanbhaipatel@gmail.com/Practice Folder/lakehouse-practice-project/src/ingestion/bronze.py:51
    from src.utils.audit import *
                                ^
SyntaxError: import * only allowed at module level


In [0]:
# spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}.{LOGS_VOLUME}")

In [0]:
# # # Imports
# # from datetime import datetime

# # Capture the start time 
# start_time = datetime.now()

# # import sys
# # import importlib
# # sys.path.append("/Workspace/Users/krishnasumanbhaipatel@gmail.com/Practice Folder/lakehouse-practice-project")

# # from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
# # from pyspark.sql.functions import * 
# # from pyspark.sql.types import * 
# # from functools import reduce
# # from operator import or_

# # import src.utils.logger
# # importlib.reload(src.utils.logger)
# # # from src.utils.logger import copy_logs_to_uc_volume 
# # from src.utils.logger import get_logger 

# # # Create a logger 
# # # logger = get_logger("bronze.customers") 
# # # logger = get_logger("bronze.customers", log_to_file=True, log_dir="/Volumes/catalog_project1/bronze/logs")
# logger = get_logger("bronze.customers", log_to_file=True)

# # # Config variables
# # CATALOG = "catalog_project1"
# # SOURCE_SCHEMA = "source1"
# # BRONZE_SCHEMA = "bronze"

# # Table name
# SOURCE_TABLE = "customers_raw"
# BRONZE_TABLE = "customers" 
# AUDIT_TABLENAME = "pipeline_runs"

# # # Source path
# # # path = "catalog_project1.source1.customers_raw"

Empty Source Table Check 

In [0]:
%skip 
if spark.read.table(f"{CATALOG}.{SOURCE_SCHEMA.lower()}.{SOURCE_TABLE}").count() == 0:
    print("Source table is empty. Skipping Bronze load.")
    # Stop processing 
else: 
    print("Pipeline has started...")

Read data, Check schema, Display sample 

Metadata _ingestion_timestamp, _source_file 

Write: mode = Overwrite, format = Delta, saveAsTable()

Verification: count(), schema, display()

In [0]:
%skip
try: 
    logger.info("Reading source table")
    cust_raw = spark.read.table(f"{CATALOG}.{SOURCE_SCHEMA.lower()}.{SOURCE_TABLE}")
    rows_read = cust_raw.count()
    logger.info(f"Rows read: {rows_read}")
    cust_raw.printSchema()

    cust_raw.show() 
    logger.info("Source ingestion completed successfully")

    logger.info("Starting Bronze ingestion")
    logger.info(f"Source table: {SOURCE_TABLE}")
    logger.info(f"Target table: {BRONZE_TABLE}")

    cust_bronze = (
        cust_raw
            .withColumn("_ingestion_timestamp", current_timestamp()) 
            .withColumn("_source_file", col("_metadata.file_path")) 
            .withColumn("_source_table", lit(f"{CATALOG}.{SOURCE_SCHEMA}.{SOURCE_TABLE}"))
    ) 

    cust_bronze.show(truncate = False)

    spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}")
    cust_bronze.write.mode("overwrite")\
                     .format("delta")\
                     .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}")

    logger.info("Bronze table written successfully.") 

    customers = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}")

    customers.select("customer_id", "_source_table").show(truncate = False)

    rows_written = spark.read.table(f"{CATALOG}.{SOURCE_SCHEMA.lower()}.{SOURCE_TABLE}").count()
    logger.info(f"Rows read: {rows_written}")

    # Check for duplicate records 
    if customers.count() == customers.dropDuplicates().count():
        print("All records are unique")
    else: 
        print("Data contains duplicate records")

    # Display duplicate records 
    display(customers.groupBy("customer_id").count().filter(col("count") > 1))

    duplicate_count = customers.groupBy("customer_id").count().filter(col("count") > 1).count()
    logger.info(f"Duplicate customer_ids: {duplicate_count}")

    # Print rows that contains null records 
    null_condition = reduce(or_, [col(c).isNull() for c in customers.columns])
    display(customers.filter(null_condition)) 

    customers.printSchema() 

    display(customers)
    # displa(customers)
    logger.info("Bronze customer ingestion completed.")

    status = "SUCCESS"
    error_message = None

except Exception as e: 
    logger.exception("Bronze ingestion failed") 
    status = "FAILED"
    error_message = str(e)
    # end_time = datetime.now()
    raise 

finally: 
    end_time = datetime.now()

In [0]:
# try: 
#     logger.info("Reading source table")
#     cust_raw = spark.read.table(f"{CATALOG}.{SOURCE_SCHEMA.lower()}.{SOURCE_TABLE}")
#     source_empty(cust_raw)
#     rows_read = get_rows_read_count(cust_raw)
#     logger.info(f"Rows read: {rows_read}")
#     cust_raw.printSchema()

#     cust_raw.show() 
#     logger.info("Source ingestion completed successfully")

#     logger.info("Starting Bronze ingestion")
#     logger.info(f"Source table: {SOURCE_TABLE}")
#     logger.info(f"Target table: {BRONZE_TABLE}")

#     # cust_bronze = (
#     #     cust_raw
#     #         .withColumn("_ingestion_timestamp", current_timestamp()) 
#     #         .withColumn("_source_file", col("_metadata.file_path")) 
#     #         .withColumn("_source_table", lit(f"{CATALOG}.{SOURCE_SCHEMA}.{SOURCE_TABLE}"))
#     # ) 

#     # cust_bronze.show(truncate = False)

#     # spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}")
#     # cust_bronze.write.mode("overwrite")\
#     #                  .format("delta")\
#     #                  .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}")

#     add_metadata(cust_raw, "cust_bronze", CATALOG, SOURCE_SCHEMA, SOURCE_TABLE, BRONZE_SCHEMA, BRONZE_TABLE)
#     logger.info("Bronze table written successfully.") 

#     customers = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}")

#     customers.select("customer_id", "_source_table").show(truncate = False)

#     rows_written = get_rows_written(customers)
#     logger.info(f"Rows written: {rows_written}")

#     # Check for duplicate records 
#     if customers.count() == customers.dropDuplicates().count():
#         print("All records are unique")
#     else: 
#         print("Data contains duplicate records")

#     # Display duplicate records 
#     display(customers.groupBy("customer_id").count().filter(col("count") > 1))

#     duplicate_count = get_duplicate_records(customers, "customer_id")
#     logger.info(f"Duplicate customer_ids: {duplicate_count}")

#     # Print rows that contains null records 
#     null_condition = reduce(or_, [col(c).isNull() for c in customers.columns])
#     display(customers.filter(null_condition)) 

#     # get Null Count 
#     null_count = get_null_count(customers)
#     customers.printSchema() 

#     display(customers)
#     # displa(customers)
#     logger.info("Bronze customer ingestion completed.")

#     status = "SUCCESS"
#     error_message = None

# except Exception as e: 
#     logger.exception("Bronze ingestion failed") 
#     status = "FAILED"
#     error_message = str(e)
#     # end_time = datetime.now()
#     raise 

# finally: 
#     end_time = datetime.now()

Verify log file exists after logging from Unity Catalog Volume 

In [0]:
%skip
files = dbutils.fs.ls("/Volumes/catalog_project1/bronze/logs/")
for file_info in files:
    print(f"File: {file_info.name}, Size: {file_info.size} bytes")

Read log file content directly 

In [0]:
%skip
log_file_path = "/Volumes/catalog_project1/bronze/logs/bronze_customers_20260714.log"

try:
    with open(log_file_path, 'r') as f:
        content = f.read()
    print(f"✓ Log file exists at: {log_file_path}")
    print(f"\nFile content ({len(content)} characters):\n")
    print(content)
except FileNotFoundError:
    print(f"✗ Log file not found at: {log_file_path}")
    print("\nChecking what files exist in the logs volume...")
    files = dbutils.fs.ls("/Volumes/catalog_project1/bronze/logs/")
    if files:
        for f in files:
            print(f"  - {f.name} ({f.size} bytes)")
    else:
        print("  No files found in /Volumes/catalog_project1/bronze/logs/")

#### Test writing to Unity Catalog Volume with python 

In [0]:
%skip
test_path = "/Volumes/catalog_project1/bronze/logs/test.txt"

try:
    with open(test_path, 'w') as f:
        f.write("Test content\n")
    print(f"✓ Successfully wrote to: {test_path}")
    
    with open(test_path, 'r') as f:
        content = f.read()
    print(f"✓ Successfully read: {content}")
    
except Exception as e:
    print(f"✗ Error: {e}")
    print(f"\nUC Volumes may not support standard Python file operations directly.")
    print(f"We need to use dbutils.fs.put() instead for UC Volumes.")

#### Check if log file exists in Unity Catalog Volume 

In [0]:
%skip
import os

log_path = "/Volumes/catalog_project1/bronze/logs/bronze_customers_20260714.log"

if os.path.exists(log_path):
    print(f"✓ Log file exists: {log_path}")
    file_size = os.path.getsize(log_path)
    print(f"  File size: {file_size} bytes")
    
    print(f"\nLog content:")
    with open(log_path, 'r') as f:
        content = f.read()
        print(content)
else:
    print(f"✗ Log file not found: {log_path}")

The leading underscore (_) in column names like `_ingestion_timestamp`, `_source_file`, and `_source_table` is a naming convention used to distinguish metadata columns from business data columns.

When want to Select only business columns (exclude underscore-prefixed)
`business_cols = [c for c in df.columns if not c.startswith('_')]`

Logging configuration: `console_handler = logging.StreamHandler(sys.stdout)` 

Means logs are: 
- ✅ Displayed in notebook cell output when cells execute
- ❌ NOT saved to any file
- ❌ NOT persisted after notebook session ends
- ❌ Lost when cell output is cleared

These appear inline with the cell output but disappear when:
- You clear the cell output
- The notebook session ends
- The cluster restarts


For Persistent logs - 
Add a FileHandler to write logs to a file:

1. Console only - 

`logger = get_logger("bronze.customers")` 

2. Console + file - 

`logger = get_logger("bronze.customers", log_to_file=True)`
> Logs written to: /tmp/logs/bronze_customers_20260714.log

3. Custom log directory - 

`logger = get_logger("bronze.customers", log_to_file=True, log_dir="/dbfs/mnt/logs")` 

- One can Unity catalog Volumns, DBFS, or any external logging services  
i. Unity Catalog Volumes (persistent storage):

`log_dir = "/Volumes/catalog_project1/bronze/logs"`

`logger = get_logger("bronze.customers", log_to_file=True, log_dir=log_dir)` 

ii. DBFS (Databricks File System):

`log_dir = "/dbfs/FileStore/logs"`

`logger = get_logger("bronze.customers", log_to_file=True, log_dir=log_dir)`


ex. For persistent storage (recommended):

##### Option 1: Unity Catalog Volume
logger = get_logger("bronze.customers", log_to_file=True, 
                   log_dir="/Volumes/catalog_project1/bronze/logs")

##### Option 2: DBFS
logger = get_logger("bronze.customers", log_to_file=True, 
                   log_dir="/dbfs/FileStore/logs")

### Create an Audit table 

In [0]:
%skip
%sql
CREATE SCHEMA IF NOT EXISTS catalog_project1.audit;

In [0]:
%skip
%sql
CREATE TABLE IF NOT EXISTS catalog_project1.audit.pipeline_runs
(
    pipeline_name STRING,
    table_name STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    rows_read BIGINT,
    rows_written BIGINT,
    status STRING,
    error_message STRING
)
USING DELTA;

In [0]:
%skip
# Creating a python audit record dictionary 
audit_record = [{
    "pipeline_name": "bronze_ingestion",
    "table_name": "customers",
    "start_time": start_time,
    "end_time": end_time,
    "rows_read": rows_read,
    "rows_written": rows_written,
    "status": status,
    "error_message": error_message
}]

# Define schema explicitly to handle None values
# from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType

audit_schema = StructType([
    StructField("pipeline_name", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("rows_read", LongType(), True),
    StructField("rows_written", LongType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True)
])

# Convert to a spark dataframe with explicit schema
audit_df = spark.createDataFrame(audit_record, schema=audit_schema)

# Append to the audit table 
audit_df.write \
    .mode("append") \
    .saveAsTable("catalog_project1.audit.pipeline_runs")

In [0]:
# write_audit_log("bronze_ingestion", "customers", start_time, end_time, rows_read, rows_written, status, error_message, CATALOG, AUDIT_SCHEMA, AUDIT_TABLENAME)

In [0]:
%skip
# Copy logs from /tmp to Unity Catalog Volume
if copy_logs_to_uc_volume(logger):
    print("✓ Logs successfully copied to UC Volume")
else:
    print("ℹ No UC Volume configured or copy failed")

In [0]:
# # %sql
# # -- SELECT * FROM CATALOG.AUDIT_SCHEMA.AUDIT_TABLENAME ORDER BY end_time DESC; 
# display(spark.sql(f"SELECT * FROM {CATALOG}.{AUDIT_SCHEMA}.{AUDIT_TABLENAME} ORDER BY end_time DESC")) 

### Querying Lakehouse System Tables 

In [0]:
%skip
%sql
SHOW SCHEMAS IN system; 

In [0]:
%skip
%sql
SHOW TABLES IN system.lakeflow; 

In [0]:
%skip
%sql
SELECT jobs.workspace_id, jobs.name, jobs.job_id, job_task_run_timeline.period_start_time, job_task_run_timeline.period_end_time, job_task_run_timeline.task_key, job_task_run_timeline.result_state FROM system.lakeflow.jobs INNER JOIN system.lakeflow.job_task_run_timeline ON jobs.job_id = job_task_run_timeline.job_id WHERE lower(jobs.name) LIKE 'demo%' ORDER BY job_task_run_timeline.period_start_time;

### Redesign the Audit Table 

In [0]:
# Later on

Verify log files exists after logging in logs folder 

In [0]:
# files = dbutils.fs.ls("/Workspace/Users/krishnasumanbhaipatel@gmail.com/Practice Folder/lakehouse-practice-project/logs")
# for file_info in files:
#     print(f"File: {file_info.name}, Size: {file_info.size} bytes")

Read log file directly 

In [0]:
# log_file_path = "/Workspace/Users/krishnasumanbhaipatel@gmail.com/Practice Folder/lakehouse-practice-project/logs/bronze_customers_20260715.log"

# try:
#     with open(log_file_path, 'r') as f:
#         content = f.read()
#     print(f"✓ Log file exists at: {log_file_path}")
#     print(f"\nFile content ({len(content)} characters):\n")
#     print(content)
# except FileNotFoundError:
#     print(f"✗ Log file not found at: {log_file_path}")
#     print("\nChecking what files exist in the logs volume...")
#     files = dbutils.fs.ls("/Volumes/catalog_project1/bronze/logs/")
#     if files:
#         for f in files:
#             print(f"  - {f.name} ({f.size} bytes)")
#     else:
#         print("No files found in /Workspace/Users/krishnasumanbhaipatel@gmail.com/Practice Folder/lakehouse-practice-project/logs/")

# After code refactoring 

In [0]:
import sys
import importlib

# Add project path to sys.path
project_path = "/Workspace/Users/krishnasumanbhaipatel@gmail.com/Practice Folder/lakehouse-practice-project"
if project_path not in sys.path:
    sys.path.insert(0, project_path)

# Import and reload to pick up sys.path changes
import src.utils.config
importlib.reload(src.utils.config)
from src.utils.config import CATALOG, BRONZE_SCHEMA, LOGS_VOLUME, SOURCE_SCHEMA, AUDIT_SCHEMA, AUDIT_TABLENAME

import src.ingestion.bronze
importlib.reload(src.ingestion.bronze)
from src.ingestion.bronze import ingest_table

ingest_table(source = "customers_raw", 
             target = "customers", 
             pipeline_name = "bronze_ingestion_pipeline", 
             AUDIT_TABLENAME = AUDIT_TABLENAME,
             CATALOG = CATALOG, 
             BRONZE_SCHEMA = BRONZE_SCHEMA, 
             LOGS_VOLUME = LOGS_VOLUME, 
             SOURCE_SCHEMA = SOURCE_SCHEMA, 
             AUDIT_SCHEMA = AUDIT_SCHEMA, 
             primary_key = "customer_id"
            )

2026-07-21 06:11:59,138 | INFO     | bronze.customers | Logging to file: /Workspace/Users/krishnasumanbhaipatel@gmail.com/Practice Folder/lakehouse-practice-project/logs/bronze_customers_20260721.log
2026-07-21 06:11:59,141 | INFO     | bronze.customers | Reading source table
Pipeline has started...
2026-07-21 06:12:20,464 | INFO     | bronze.customers | Rows read: 10
root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- signup_date: string (nullable = true)

2026-07-21 06:12:21,453 | INFO     | bronze.customers | Source fetching completed successfully
2026-07-21 06:12:21,456 | INFO     | bronze.customers | Starting Bronze ingestion
2026-07-21 06:12:21,456 | INFO     | bronze.customers | Source table: customers_raw
2026-07-21 06:12:21,457 | INFO     | bronze.customers | Target table: customers
2026-07-21 06:12:37,777 | INFO     | bronze.custo

'Pipeline ran successfully'

In [0]:
spark.sql(f"SELECT * FROM catalog_project1.audit.pipeline_runs ORDER BY start_time DESC").show()

+--------------------+----------+--------------------+--------------------+---------+------------+-------+--------------------+
|       pipeline_name|table_name|          start_time|            end_time|rows_read|rows_written| status|       error_message|
+--------------------+----------+--------------------+--------------------+---------+------------+-------+--------------------+
|bronze_ingestion_...| customers|2026-07-21 06:11:...|2026-07-21 06:12:...|       10|          10|SUCCESS|                NULL|
|bronze_ingestion_...|  products|2026-07-15 13:37:...|2026-07-15 13:38:...|       10|          10|SUCCESS|                NULL|
|bronze_ingestion_...|  products|2026-07-15 13:28:...|2026-07-15 13:28:...|       10|          10|SUCCESS|                NULL|
|bronze_ingestion_...| customers|2026-07-15 12:52:...|2026-07-15 12:52:...|       10|          10|SUCCESS|                NULL|
|    bronze_ingestion| customers|2026-07-15 10:09:...|2026-07-15 10:10:...|       10|          10|SUCCES

In [0]:
spark.read.table("catalog_project1.bronze.customers").toPandas()

,customer_id,name,email,city,state,signup_date,_ingestion_timestamp,_source_file,_source_table,_ingestion_job_id,_ingestion_pipeline_id
0,1001,Rahul Shah,rahul.shah@email.com,Ahmedabad,Gujarat,10-01-2025,2026-07-21 06:12:28.207206,s3://dbstorage-prod-sul6n/uc/1304586b-ace6-48c...,catalog_project1.source1.customers_raw,None,None
1,1002,Priya Patel,priya.patel@email.com,Surat,Gujarat,12-01-2025,2026-07-21 06:12:28.207206,s3://dbstorage-prod-sul6n/uc/1304586b-ace6-48c...,catalog_project1.source1.customers_raw,None,None
2,1003,Amit Verma,amit.verma@email.com,Delhi,Delhi,15-01-2025,2026-07-21 06:12:28.207206,s3://dbstorage-prod-sul6n/uc/1304586b-ace6-48c...,catalog_project1.source1.customers_raw,None,None
3,1004,Sneha Iyer,sneha.iyer@email.com,Bengaluru,Karnataka,18-01-2025,2026-07-21 06:12:28.207206,s3://dbstorage-prod-sul6n/uc/1304586b-ace6-48c...,catalog_project1.source1.customers_raw,None,None
4,1005,Rohan Gupta,rohan.gupta@email.com,Mumbai,Maharashtra,20-01-2025,2026-07-21 06:12:28.207206,s3://dbstorage-prod-sul6n/uc/1304586b-ace6-48c...,catalog_project1.source1.customers_raw,None,None
5,1006,Neha Sharma,neha.sharma@email.com,Jaipur,Rajasthan,22-01-2025,2026-07-21 06:12:28.207206,s3://dbstorage-prod-sul6n/uc/1304586b-ace6-48c...,catalog_project1.source1.customers_raw,None,None
6,1007,Karan Mehta,karan.mehta@email.com,Pune,Maharashtra,24-01-2025,2026-07-21 06:12:28.207206,s3://dbstorage-prod-sul6n/uc/1304586b-ace6-48c...,catalog_project1.source1.customers_raw,None,None
7,1008,Anjali Singh,anjali.singh@email.com,Lucknow,Uttar Pradesh,27-01-2025,2026-07-21 06:12:28.207206,s3://dbstorage-prod-sul6n/uc/1304586b-ace6-48c...,catalog_project1.source1.customers_raw,None,None
8,1009,Vikram Joshi,vikram.joshi@email.com,Indore,Madhya Pradesh,29-01-2025,2026-07-21 06:12:28.207206,s3://dbstorage-prod-sul6n/uc/1304586b-ace6-48c...,catalog_project1.source1.customers_raw,None,None
9,1010,Meera Nair,meera.nair@email.com,Kochi,Kerala,01-02-2025,2026-07-21 06:12:28.207206,s3://dbstorage-prod-sul6n/uc/1304586b-ace6-48c...,catalog_project1.source1.customers_raw,None,None
